# K-Means Clustering: Algorithm, Mathematics, & Implementation

**K-Means Clustering** is one of the most popular and fundamental unsupervised machine learning algorithms. It is a centroid-based, partitioning clustering method that groups a dataset of unlabeled points into $K$ distinct, non-overlapping clusters. The algorithm works iteratively to assign each data point to its nearest centroid while updating centroids to represent the average of all points assigned to them.

In these detailed notes, we will cover:
1. **Core Concept:** Centroid-based partitioning, Lloyd's algorithm steps (Initialization, Assignment, Update), convergence criteria, and key limitations.
2. **Mathematical Formulation (Inertia/WCSS Optimization):** The math of the objective function, the analytical derivation of centroid updates, and the K-Means++ initialization algorithm.
3. **Choosing the Optimal K:** The Elbow Method and Silhouette Analysis (cohesion vs. separation).
4. **Key Hyperparameters:** Understanding `n_clusters`, `init`, `n_init`, `max_iter`, and `tol` in Scikit-Learn.
5. **Practical Python Demos:**
   - **Demo A:** Step-by-step implementation of `KMeansScratch` from scratch, displaying convergence pathways.
   - **Demo B:** Visualizing the vulnerability of random initialization to local minima and how K-Means++ solves it.
   - **Demo C:** Training & tuning `KMeans` using `scikit-learn` along with Silhouette Analysis visualizations.
   - **Summary Checklist** for K-Means Clustering.

## 1. Core Concepts & Lloyd's Algorithm

The primary goal of K-Means is to divide $n$ observations into $K$ clusters in which each observation belongs to the cluster with the nearest mean (centroid). 

### Lloyd's Algorithm
The standard heuristic algorithm for solving K-Means is **Lloyd's Algorithm**, which iterates through three core steps:

1. **Centroid Initialization:** Choose $K$ initial centroids $\{\mu_1, \mu_2, \dots, \mu_k\}$ in the feature space.
2. **Cluster Assignment:** Assign each data point $x_i$ to the closest centroid based on the squared Euclidean distance:
   $$c^{(i)} := \arg\min_{j} \|x_i - \mu_j\|^2$$
3. **Centroid Update:** Recalculate each centroid $\mu_j$ as the mean of all points assigned to cluster $j$:
   $$\mu_j := \frac{\sum_{i=1}^n \mathbb{I}(c^{(i)} = j) x_i}{\sum_{i=1}^n \mathbb{I}(c^{(i)} = j)}$$
   where $\mathbb{I}(\cdot)$ is the indicator function.

These steps are repeated until **convergence criteria** are met. Convergence is achieved when:
- Centroid positions no longer change (or change is below a threshold $\text{tol}$).
- Point assignments to clusters remain completely unchanged.
- The maximum number of iterations (`max_iter`) is reached.

### Key Limitations of K-Means
- **Vulnerability to Initialization:** The algorithm is a greedy search and can easily get trapped in sub-optimal local minima depending on initial centroid positions.
- **Assumes Spherical Clusters:** K-Means uses Euclidean distance, implying clusters are spherical, isotropic, and roughly equal in size. It performs poorly on elongated, nested, or complex geometric structures (e.g., moons, concentric circles).
- **Vulnerable to Varying Sizes/Densities:** If clusters have highly unequal densities or sizes, K-Means often splits large clusters or absorbs small clusters into larger ones.
- **Sensitivity to Outliers:** Since centroids are calculated using arithmetic means, a single outlier can significantly pull a centroid away from the actual center of the cluster.

## 2. Mathematical Formulation: Inertia Optimization

K-Means formulation is framed as minimizing the **Within-Cluster Sum of Squares (WCSS)**, also referred to as **Inertia**:

$$J(C, \mu) = \sum_{i=1}^{n} \sum_{j=1}^{K} r_{ij} \|x_i - \mu_j\|^2$$

Where:
- $x_i \in \mathbb{R}^d$: The $i$-th data point ($n$ samples total).
- $\mu_j \in \mathbb{R}^d$: The centroid of cluster $j$ ($K$ clusters total).
- $r_{ij} \in \{0, 1\}$: Binary indicator variable representing cluster membership. $r_{ij} = 1$ if $x_i$ is assigned to cluster $j$, and $r_{ij} = 0$ otherwise.

Since $x_i$ must belong to exactly one cluster, we have $\sum_{j=1}^K r_{ij} = 1$ for all $i$.

---

### Step 1: Solving for Cluster Assignment (fixing $\mu_j$)
If we fix the centroids $\mu_j$, we minimize $J$ with respect to the assignments $r_{ij}$. Since the samples are independent, we minimize the term for each sample $i$ individually:

$$r_{ij} = \begin{cases} 1 & \text{if } j = \arg\min_k \|x_i - \mu_k\|^2 \\ 0 & \text{otherwise} \end{cases}$$

This mathematically justifies the **Cluster Assignment** step using Euclidean distance.

---

### Step 2: Solving for Centroids (fixing $r_{ij}$)
If we fix the cluster assignments $r_{ij}$, we find the optimal centroids $\mu_j$ by setting the partial derivative of $J$ with respect to $\mu_j$ to zero:

$$\frac{\partial J}{\partial \mu_j} = \frac{\partial}{\partial \mu_j} \sum_{i=1}^n r_{ij} \|x_i - \mu_j\|^2 = \sum_{i=1}^n r_{ij} \cdot 2(\mu_j - x_i) = 0$$

$$2 \sum_{i=1}^n r_{ij} \mu_j - 2 \sum_{i=1}^n r_{ij} x_i = 0$$

$$\mu_j \sum_{i=1}^n r_{ij} = \sum_{i=1}^n r_{ij} x_i$$

$$\mu_j^* = \frac{\sum_{i=1}^n r_{ij} x_i}{\sum_{i=1}^n r_{ij}}$$

This derivation proves that the centroid update rule (computing the arithmetic mean of points in the cluster) analytically minimizes the WCSS objective.

---

### K-Means++ Initialization Algorithm
To solve the local minima vulnerability, **K-Means++** initializes centroids spread out across the dataset. The algorithm steps are:
1. **Select First Centroid:** Choose one centroid $\mu_1$ uniformly at random from the dataset $X$.
2. **Compute Distances:** For each data point $x \in X$, compute the squared Euclidean distance to the nearest centroid already selected:
   $$D(x)^2 = \min_{c \in \text{Centroids}} \|x - c\|^2$$
3. **Probabilistic Selection:** Choose the next centroid $\mu_k$ from $X$ with a probability distribution proportional to the squared distances:
   $$P(x) = \frac{D(x)^2}{\sum_{y \in X} D(y)^2}$$
4. **Repeat:** Repeat steps 2 and 3 until $K$ centroids are chosen.

## 3. Choosing the Optimal K & Hyperparameters

Unsupervised learning lacks labels, making model evaluation challenging. We evaluate clustering quality using two primary techniques:

### The Elbow Method
Plot the WCSS (Inertia) against the number of clusters $K$. 
- As $K$ increases, WCSS always decreases (at $K=n$, WCSS is 0).
- We look for the "elbow" point—the point where the rate of decrease in WCSS shifts from steep to shallow. This indicates that adding more clusters yields diminishing returns in explaining data variance.

### Silhouette Coefficient
Measures how similar an object is to its own cluster (cohesion) compared to other clusters (separation). For a point $i$:
- $a(i)$: The average distance between $i$ and all other points in the same cluster.
- $b(i)$: The average distance between $i$ and all points in the nearest neighboring cluster.

The Silhouette Coefficient $s(i)$ is defined as:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

- $s(i) \approx 1$: Point is well matched to its cluster and far from neighboring clusters.
- $s(i) \approx 0$: Point is on the decision boundary between two clusters.
- $s(i) \approx -1$: Point is likely assigned to the wrong cluster.

The overall **Silhouette Score** is the mean silhouette coefficient across all samples.

---

### Key Hyperparameters in Scikit-Learn
- **`n_clusters` ($K$):** The number of clusters to form and centroids to generate.
- **`init`:** Centroid initialization method. `'k-means++'` (standard default) or `'random'`.
- **`n_init`:** Number of times the algorithm runs with different centroid seeds. The best run (lowest WCSS) is kept. Crucial for overcoming bad initialization in `'random'` mode.
- **`max_iter`:** Maximum iterations of the algorithm for a single run (default is 300).
- **`tol`:** Relative tolerance of WCSS change to declare convergence (default is $10^{-4}$).

## 4. Code Setup & Imports

Let's prepare the Python environment and import the required libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
np.random.seed(42)

print("Libraries imported successfully!")

## 5. Demo A: K-Means Implementation From Scratch

We will build a custom class `KMeansScratch` to implement Lloyd's algorithm with support for both random and K-Means++ initializations. We will track the history of centroid coordinates at each step to visualize convergence.

In [ ]:
class KMeansScratch:
    def __init__(self, n_clusters=3, init='k-means++', max_iter=300, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.init = init
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia_ = None
        self.history = [] # To store centroid history

    def fit(self, X):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        self.history = []
        
        # 1. Initialize Centroids
        if self.init == 'k-means++':
            self.centroids = self._init_kmeans_pp(X)
        else:
            self.centroids = self._init_random(X)
            
        self.history.append(self.centroids.copy())
        
        for iteration in range(self.max_iter):
            old_centroids = self.centroids.copy()
            
            # 2. Cluster Assignment
            self.labels = self._assign_clusters(X)
            
            # 3. Centroid Update
            self.centroids = self._update_centroids(X, self.labels)
            self.history.append(self.centroids.copy())
            
            # Check convergence
            centroid_shift = np.sum((self.centroids - old_centroids) ** 2)
            if centroid_shift < self.tol:
                break
                
        # Compute final WCSS (inertia)
        self.inertia_ = np.sum((X - self.centroids[self.labels]) ** 2)
        return self

    def _init_random(self, X):
        idx = np.random.choice(X.shape[0], self.n_clusters, replace=False)
        return X[idx].copy()

    def _init_kmeans_pp(self, X):
        n_samples, n_features = X.shape
        centroids = np.empty((self.n_clusters, n_features))
        
        # Choose first centroid uniformly at random
        first_idx = np.random.choice(n_samples)
        centroids[0] = X[first_idx]
        
        for k in range(1, self.n_clusters):
            # Compute distance squared to the nearest existing centroid
            distances = np.min([np.sum((X - c) ** 2, axis=1) for c in centroids[:k]], axis=0)
            
            # Select next centroid with probability proportional to D(x)^2
            probs = distances / np.sum(distances)
            next_idx = np.random.choice(n_samples, p=probs)
            centroids[k] = X[next_idx]
            
        return centroids

    def _assign_clusters(self, X):
        # Compute distance matrix between X (n_samples, n_features) and centroids (n_clusters, n_features)
        distances = np.sqrt(np.sum((X[:, np.newaxis] - self.centroids) ** 2, axis=2))
        return np.argmin(distances, axis=1)

    def _update_centroids(self, X, labels):
        centroids = np.empty((self.n_clusters, X.shape[1]))
        for k in range(self.n_clusters):
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                centroids[k] = np.mean(cluster_points, axis=0)
            else:
                centroids[k] = self.centroids[k] # Hold position if empty
        return centroids

    def predict(self, X):
        distances = np.sqrt(np.sum((X[:, np.newaxis] - self.centroids) ** 2, axis=2))
        return np.argmin(distances, axis=1)

# --- Test the Scratch Implementation ---
X_toy, y_toy = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=42)

# Fit model
model = KMeansScratch(n_clusters=4, init='k-means++', random_state=42)
model.fit(X_toy)

print(f"Scratch WCSS (Inertia): {model.inertia_:.4f}")
print(f"Converged in {len(model.history)-1} iterations.")

### Visualizing Convergence Pathway
Let's visualize the points colored by cluster alongside the path of the centroids moving from iteration 0 to convergence.

In [ ]:
plt.figure(figsize=(10, 7))
# Plot cluster points
colors = ['#1abc9c', '#3498db', '#9b59b6', '#e67e22']
for k in range(model.n_clusters):
    cluster_data = X_toy[model.labels == k]
    plt.scatter(cluster_data[:, 0], cluster_data[:, 1], c=colors[k], alpha=0.6, edgecolors='black', label=f'Cluster {k}')

# Convert history to array shape: (n_iterations, n_clusters, n_features)
hist_arr = np.array(model.history)

# Plot centroid trajectories
for k in range(model.n_clusters):
    traj = hist_arr[:, k, :]
    plt.plot(traj[:, 0], traj[:, 1], color='black', linestyle='--', linewidth=1.5)
    plt.scatter(traj[:-1, 0], traj[:-1, 1], color='grey', marker='x', s=60, zorder=5)
    plt.scatter(traj[-1, 0], traj[-1, 1], color='red', marker='*', s=200, edgecolors='black', zorder=10, label='Final Centroid' if k==0 else "")

plt.title("K-Means Scratch Convergence Paths", fontsize=14, fontweight='bold')
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

## 6. Demo B: Initialization Vulnerability (Random vs. K-Means++)

Random centroid selection often leads to sub-optimal groupings when two initial seeds fall in the same natural cluster. K-Means++ addresses this by selecting centroids that are far apart.

Let's create a scenario with a random seed where `init='random'` gets stuck in a local minimum, and compare it to `init='k-means++'`.

In [ ]:
# Set a specific seed that forces random initialization to fail
bad_seed = 50

kmeans_random = KMeansScratch(n_clusters=4, init='random', random_state=bad_seed)
kmeans_random.fit(X_toy)

kmeans_pp = KMeansScratch(n_clusters=4, init='k-means++', random_state=bad_seed)
kmeans_pp.fit(X_toy)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Random Initialization Results
axes[0].scatter(X_toy[:, 0], X_toy[:, 1], c=kmeans_random.labels, cmap='viridis', alpha=0.6, edgecolors='black')
axes[0].scatter(kmeans_random.centroids[:, 0], kmeans_random.centroids[:, 1], c='red', marker='*', s=250, edgecolors='black', label='Final Centroid')
axes[0].set_title(f"Random Initialization (Inertia: {kmeans_random.inertia_:.2f})\n Stuck in Local Minimum", fontsize=12, fontweight='bold')
axes[0].legend()

# Plot K-Means++ Results
axes[1].scatter(X_toy[:, 0], X_toy[:, 1], c=kmeans_pp.labels, cmap='viridis', alpha=0.6, edgecolors='black')
axes[1].scatter(kmeans_pp.centroids[:, 0], kmeans_pp.centroids[:, 1], c='red', marker='*', s=250, edgecolors='black', label='Final Centroid')
axes[1].set_title(f"K-Means++ Initialization (Inertia: {kmeans_pp.inertia_:.2f})\n Converged Globally", fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle("Effect of Initialization Strategies on K-Means", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Demo C: Scikit-Learn Tuning & Silhouette Analysis

We will now use the official `scikit-learn` library. We will evaluate our model selection by plotting the **Elbow Curve** and drawing a **Silhouette Diagram**.

In [ ]:
# Compute inertia and silhouette scores for K ranging from 2 to 8
k_range = range(2, 9)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42)
    labels = km.fit_predict(X_toy)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_toy, labels))

# Plot Elbow and Silhouette Scores
fig, ax1 = plt.subplots(figsize=(10, 5))

color = '#e74c3c'
ax1.set_xlabel('Number of Clusters (K)', fontweight='bold')
ax1.set_ylabel('Inertia / WCSS', color=color, fontweight='bold')
ax1.plot(k_range, inertias, marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_title("Elbow Method and Silhouette Scores for K Selection", fontsize=13, fontweight='bold')

ax2 = ax1.twinx()
color = '#2ecc71'
ax2.set_ylabel('Silhouette Score', color=color, fontweight='bold')
ax2.plot(k_range, sil_scores, marker='s', color=color, linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.show()

### Silhouette Diagram for $K=3$ vs $K=4$
A silhouette diagram plots the sorted silhouette values for each sample in each cluster. It lets us check if clusters are of similar thickness and if all clusters pass the average score.

In [ ]:
for n_clusters in [3, 4]:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Fit and compute scores
    km = KMeans(n_clusters=n_clusters, init='k-means++', random_state=42)
    cluster_labels = km.fit_predict(X_toy)
    avg_sil_score = silhouette_score(X_toy, cluster_labels)
    sample_silhouette_values = silhouette_samples(X_toy, cluster_labels)
    
    y_lower = 10
    for i in range(n_clusters):
        # Aggregate silhouette values for samples in cluster i, and sort them
        ith_cluster_sil_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_sil_values.sort()
        
        size_cluster_i = ith_cluster_sil_values.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.nipy_spectral(float(i) / n_clusters)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                          0, ith_cluster_sil_values,
                          facecolor=color, edgecolor=color, alpha=0.7)
        
        # Label the silhouette plots with cluster numbers in the middle
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10 # 10 for the 0 samples gap between clusters
        
    ax.set_title(f"Silhouette Plot for K={n_clusters} (Avg Silhouette: {avg_sil_score:.3f})", fontsize=13, fontweight='bold')
    ax.set_xlabel("Silhouette Coefficient Values")
    ax.set_ylabel("Cluster Label")
    
    # Draw vertical line indicating the average silhouette score
    ax.axvline(x=avg_sil_score, color="red", linestyle="--")
    ax.set_yticks([]) # Clear the yaxis labels / ticks
    ax.set_xlim([-0.1, 1])
    plt.show()

## Summary Checklist for K-Means Clustering

1. **Objective Function (Inertia):** Minimize within-cluster sum of squares:
   $$J = \sum_{i=1}^n \sum_{j=1}^K r_{ij} \|x_i - \mu_j\|^2$$
2. ** Lloyd's Algorithm Iteration:**
   - *Assignment:* Assign each point to the closest centroid ($r_{ij}$). 
   - *Update:* Move centroid to the mean of assigned points ($\mu_j = \frac{\sum r_{ij} x_i}{\sum r_{ij}}$).
3. **Centroid Initialization (K-Means++):** Prevents local minima convergence by choosing initial centroids sequentially with probability proportional to the squared distance ($D(x)^2$) to the closest existing centroid.
4. **Optimal K selection:**
   - *Elbow Method:* Find the point where inertia rate of decrease slows down drastically.
   - *Silhouette Analysis:* Check average silhouette score (aim for high scores near 1) and analyze cluster-wise silhouette diagrams for uniformity and size consistency.
5. **Core Constraints:** Assumes spherical/convex shapes, equal variances (densities), and is sensitive to outliers.